In [0]:
silver_patients=spark.read.table("workspace.silver.patients")
silver_encounters=spark.read.table("workspace.silver.encounters")
silver_conditions=spark.read.table("workspace.silver.conditions")
silver_observations=spark.read.table("workspace.silver.observations")
silver_medications=spark.read.table("workspace.silver.medications")
silver_claims=spark.read.table("workspace.silver.claims")

In [0]:
display(silver_patients)

DASHBOARD 1

In [0]:
display(silver_encounters)

In [0]:
from pyspark.sql.functions import col, when, year, current_date, datediff

dim_patient = silver_patients.select(
    col("patient_id").alias("patient_key"),
    col("gender"),
    col("dob"),
    (datediff(current_date(), col("dob")) / 365.25).cast("int").alias("age"),
    col("city"),
    col("state"),
    when(col("deceased_datetime").isNotNull(), 1).otherwise(0).alias("is_deceased"),
    when((datediff(current_date(), col("dob")) / 365.25) < 18, "Child")
        .when((datediff(current_date(), col("dob")) / 365.25) < 65, "Adult")
        .otherwise("Senior").alias("age_group")
).dropDuplicates(["patient_key"])

display(dim_patient)

In [0]:
display(silver_conditions)

In [0]:
dim_condition = (
    silver_conditions
    .select(
        col("condition_code").alias("condition_key"),
        col("condition_code"),
        col("condition_description")
    )
    .dropDuplicates(["condition_key"])
)

display(dim_condition)

In [0]:
from pyspark.sql.functions import col, year, month, date_format, quarter

dim_date = silver_encounters.select(
    date_format(col("encounter_start"), "yyyyMMdd").cast("int").alias("date_key"),
    col("encounter_start").alias("date"),
    year(col("encounter_start")).alias("year"),
    date_format(col("encounter_start"), "MMMM").alias("month_name"),
    quarter(col("encounter_start")).alias("quarter")
)

display(dim_date)

In [0]:
from pyspark.sql.functions import col, date_format, lit

fact_encounter = (
    silver_encounters
    .select(
        col("encounter_id").alias("encounter_key"),
        col("patient_id").alias("patient_key"),
        date_format(col("encounter_start"), "yyyyMMdd").cast("int").alias("date_key"),
        col("encounter_class").alias("encounter_type"),
        lit(1).alias("encounter_count")
    )
    .dropDuplicates(["encounter_key"])
)

display(fact_encounter)

In [0]:
dim_patient.write.format("delta").mode("overwrite").saveAsTable("gold.dim_patient")
dim_condition.write.format("delta").mode("overwrite").saveAsTable("gold.dim_condition")
dim_date.write.format("delta").mode("overwrite").saveAsTable("gold.dim_date")
fact_encounter.write.format("delta").mode("overwrite").saveAsTable("gold.fact_encounter")